In [1]:
import sys; sys.path.insert(0, '..')

from utils import (flag_rfi_channels, flag_outlier_dumps, vlsr_correction,
                    compute_R_for_dumps, compute_cell_metrics, neighbor_qa)

from pathlib import Path
import datetime as dt
import numpy as np
from ugradiolab import plotting

SAMPLE_RATE_HZ = 2.56e6
NFFT = 1024
HI_REST_MHZ = 1420.405
C_KMS = 299792.458

%matplotlib inline

/home/ikaros/projects/ay-121/.venv/lib/python3.12/site-packages/rtlsdr/__init__.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Load all scan dumps

In [2]:
STREAMING_DIR = Path('../../../data/lab04/streaming')

scan_dirs = []
for session_dir in sorted(STREAMING_DIR.glob('session_*')):
    obs_found = sorted(session_dir.glob('obs_*'))
    cal_found = sorted(session_dir.glob('cal_*'))
    scan_dirs.extend(obs_found)
    scan_dirs.extend(cal_found)

# Collect all dumps
records = []
for d in scan_dirs:
    session_label = d.parent.name
    for p in sorted(d.glob('*.npz')):
        with np.load(p, allow_pickle=True) as f:
            records.append({
                'path': p,
                'session': session_label,
                'target': str(f['target_name']),
                'corr00': f['corr00'].astype(float),
                'corr11': f['corr11'].astype(float),
                'lo_mhz': float(f['lo_freq_mhz']),
                'noise_on': bool(f['noise_on']),
                'time': float(f['time']),
                'alt': float(f['alt_deg']),
                'az': float(f['az_deg']),
                'ra': float(f['ra_deg']),
                'dec': float(f['dec_deg']),
            })

N = len(records)

# Compute galactic coordinates from RA/Dec metadata
import astropy.coordinates as ac
import astropy.units as u_ast

for r in records:
    c = ac.SkyCoord(ra=r['ra'] * u_ast.deg, dec=r['dec'] * u_ast.deg, frame='icrs')
    r['gl'] = round(c.galactic.l.deg)
    r['gb'] = round(c.galactic.b.deg)

sessions = sorted(set(r['session'] for r in records))
lo_unique = sorted(set(r['lo_mhz'] for r in records if not r['noise_on']))
n_cal = sum(1 for r in records if r['noise_on'])
n_sci = sum(1 for r in records if not r['noise_on'])

import pandas as pd

# Build summary DataFrame
session_data = []
for s in sessions:
    n_cells = len(sorted(set((r['gl'], r['gb']) for r in records if r['session'] == s and not r['noise_on'])))
    n_dumps = sum(1 for r in records if r['session'] == s)
    if n_dumps > 0:
        session_data.append({'Session': s, 'No. of Obs. Cells': n_cells, 'No. of Dumps': n_dumps})

session_df = pd.DataFrame(session_data)
display(session_df)

gl_vals = sorted(set(r['gl'] for r in records if r['gl'] is not None))
gb_vals = sorted(set(r['gb'] for r in records if r['gb'] is not None))
print(f'Grid: {len(gb_vals)} b-rows x {len(gl_vals)} l-cols')
print(f'LO: {lo_unique}')
print(f'Cal dumps: {n_cal}, Science dumps: {n_sci}')

,Session,No. of Obs. Cells,No. of Dumps
0,session_001,153,1225
1,session_002,14,432
2,session_003,95,758
3,session_004,67,537
4,session_005,19,149
5,session_006,97,775
6,session_007,80,641
7,session_008,75,601
8,session_009,153,1225
9,session_010,40,321


Grid: 34 b-rows x 235 l-cols
LO: [1420.0, 1421.0]
Cal dumps: 146, Science dumps: 13792


## 2. fftshift + RFI flagging + outlier dump filter

In [3]:
DC_BIN = NFFT // 2
f_bb_mhz = np.fft.fftshift(np.fft.fftfreq(NFFT, d=1.0/SAMPLE_RATE_HZ)) / 1e6
df_khz = SAMPLE_RATE_HZ / NFFT / 1e3

for r in records:
    r['corr00'] = np.fft.fftshift(r['corr00'])
    r['corr11'] = np.fft.fftshift(r['corr11'])
    r['stokes_I'] = r['corr00'] + r['corr11']

# RFI flagging: sliding-window Chebyshev pseudo-continuum + MAD sigma clip.
# Local extrema are excluded from the fit so the continuum is not biased
# by RFI spikes or dropouts.  No hard-coded DC masking is needed -- the
# Chebyshev fit naturally excludes the DC spike as a local maximum.
RFI_WINDOW = 15
RFI_SIGMA = 10.0
n_flagged = sum(flag_rfi_channels(r['stokes_I'], RFI_WINDOW, RFI_SIGMA)
                for r in records)

print(f'Baseband: [{f_bb_mhz[0]:.3f}, {f_bb_mhz[-1]:.3f}] MHz, '
      f'dnu = {df_khz:.2f} kHz')
print(f'RFI flagged: {n_flagged} samples across {N} dumps')

Baseband: [-1.280, 1.278] MHz, dnu = 2.50 kHz
RFI flagged: 93633 samples across 13938 dumps


In [4]:
SHAPE_DEV_THRESH = 0.10
SHAPE_FRAC_THRESH = 0.20

outlier_records = flag_outlier_dumps(records, SHAPE_DEV_THRESH, SHAPE_FRAC_THRESH)
n_outlier_removed = len(outlier_records)
N = len(records)

print(f'Outlier dump filter (spectral shape): removed {n_outlier_removed} dumps')
print(f'  (flag if >{SHAPE_FRAC_THRESH:.0%} of channels deviate '
      f'>{SHAPE_DEV_THRESH:.0%} from group median)')
print(f'Remaining: {N} dumps')

Outlier dump filter (spectral shape): removed 31 dumps
  (flag if >20% of channels deviate >10% from group median)
Remaining: 13907 dumps


## 3. Frequency-switched profiles per cell

For each cell, pair dumps at the two LO frequencies and compute
`R = (I1 - I2) / I2`.

**LSR correction**: Each dump's topocentric velocity is shifted to the
kinematic LSR frame (solar motion 20 km/s toward 18h +30deg B1900).
Combined results interpolate each session onto a common LSR velocity grid
before averaging, correcting for Earth's orbital motion between sessions.

In [5]:
import astropy.coordinates as ac
import astropy.units as u_ast
from astropy.time import Time as AstroTime
from collections import defaultdict

EDGE_TRIM_MHZ = 0.256
MOLL_CENTER_L = 120.0

lo1, lo2 = lo_unique[0], lo_unique[1]
f_sky = lo1 + f_bb_mhz
f_sky_2 = lo2 + f_bb_mhz
f_overlap_lo = max(f_sky[0], f_sky_2[0]) + EDGE_TRIM_MHZ
f_overlap_hi = min(f_sky[-1], f_sky_2[-1]) - EDGE_TRIM_MHZ
overlap_mask = (f_sky >= f_overlap_lo) & (f_sky <= f_overlap_hi)
f_overlap = f_sky[overlap_mask]
v_overlap = C_KMS * (1 - f_overlap / HI_REST_MHZ)  # topocentric
dv_kms = np.abs(np.median(np.diff(v_overlap)))

print(f'LO pair: ({lo1}, {lo2}) MHz')
print(f'Overlap: [{f_overlap_lo:.2f}, {f_overlap_hi:.2f}] MHz')
print(f'Velocity (topo): [{v_overlap[-1]:.0f}, {v_overlap[0]:.0f}] km/s')
print(f'dv = {dv_kms:.3f} km/s per channel')

# --- Assign galactic coords ---
for r in records:
    if r.get('gl') is None:
        r['gl'], r['gb'] = None, None
        continue
    c = ac.SkyCoord(ra=r['ra'] * u_ast.deg, dec=r['dec'] * u_ast.deg, frame='icrs')
    r['gl'] = round(c.galactic.l.deg)
    r['gb'] = round(c.galactic.b.deg)

# --- LSR velocity correction ---
# v_LSR = v_topo + v_corr, where v_corr = heliocentric + solar motion to LSRK.
# Compute one correction per (DR, cell) group -- all dumps in a group are
# taken within minutes, so share the same correction to < 0.01 km/s.
print('Computing LSR corrections...')
cell_dr_groups = defaultdict(list)
for r in records:
    if r.get('gl') is None or r['noise_on']:
        r['v_corr_lsr'] = 0.0
        continue
    cell_dr_groups[(r['session'], r['gl'], r['gb'])].append(r)

for key, group in cell_dr_groups.items():
    r0 = group[0]
    mean_t = np.mean([r['time'] for r in group])
    v_corr = vlsr_correction(r0['ra'], r0['dec'], mean_t)
    for r in group:
        r['v_corr_lsr'] = v_corr

sci_vcorr = [r['v_corr_lsr'] for r in records
             if r.get('gl') is not None and not r['noise_on']]
mean_vcorr = np.mean(sci_vcorr)
print(f'  Range: {min(sci_vcorr):.2f} to {max(sci_vcorr):.2f} km/s '
      f'(mean {mean_vcorr:.2f})')

# Common LSR velocity grid for combined results
v_lsr_overlap = v_overlap + mean_vcorr
print(f'Velocity (LSR):  [{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s')

# Per-DR LSR corrections
for dr in sessions:
    dr_vc = [r['v_corr_lsr'] for r in records
             if r['session'] == dr and r.get('gl') is not None and not r['noise_on']]
    if dr_vc:
        print(f'  {dr}: v_corr = {np.mean(dr_vc):+.2f} km/s '
              f'(shift vs ref: {np.mean(dr_vc) - mean_vcorr:+.2f})')

# --- All unique (l, b) pointings ---
all_pointings = sorted(set((r['gl'], r['gb']) for r in records
                           if r['gl'] is not None))

# --- Build results ---
cell_results = {}           # per-DR, topocentric (for per-DR plots)
cell_results_combined = {}  # combined, LSR-corrected

for gl, gb in all_pointings:
    sci_dumps = [r for r in records
                 if r['gl'] == gl and r['gb'] == gb and not r['noise_on']]

    # Combined with LSR correction
    result = compute_R_for_dumps(sci_dumps, lo1, lo2, overlap_mask, v_overlap,
                                 lsr_correct=True, v_lsr_grid=v_lsr_overlap)
    if result is not None:
        cell_results_combined[(gl, gb)] = result

    # Per DR (topocentric -- intra-session shift is negligible)
    for dr in sessions:
        dr_dumps = [r for r in sci_dumps if r['session'] == dr]
        result = compute_R_for_dumps(dr_dumps, lo1, lo2, overlap_mask, v_overlap)
        if result is not None:
            cell_results[(dr, gl, gb)] = result


LO pair: (1420.0, 1421.0) MHz
Overlap: [1419.98, 1421.02] MHz
Velocity (topo): [-130, 90] km/s
dv = 0.528 km/s per channel
Computing LSR corrections...
  Range: -43.89 to 43.33 km/s (mean -14.04)
Velocity (LSR):  [-144, 76] km/s
  session_001: v_corr = +24.45 km/s (shift vs ref: +38.50)
  session_002: v_corr = -33.58 km/s (shift vs ref: -19.54)
  session_003: v_corr = +11.25 km/s (shift vs ref: +25.30)
  session_004: v_corr = -32.89 km/s (shift vs ref: -18.85)
  session_005: v_corr = -39.99 km/s (shift vs ref: -25.95)
  session_006: v_corr = -36.02 km/s (shift vs ref: -21.97)
  session_007: v_corr = +28.55 km/s (shift vs ref: +42.59)
  session_008: v_corr = -18.46 km/s (shift vs ref: -4.42)
  session_009: v_corr = -36.79 km/s (shift vs ref: -22.75)
  session_010: v_corr = -41.64 km/s (shift vs ref: -27.60)
  session_011: v_corr = -3.54 km/s (shift vs ref: +10.51)
  session_012: v_corr = -37.16 km/s (shift vs ref: -23.12)
  session_013: v_corr = +40.71 km/s (shift vs ref: +54.76)
  sess

/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:70: RuntimeWarning: Mean of empty slice
  R_sess = np.nanmean(R_pairs, axis=0)
/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:95: RuntimeWarning: Mean of empty slice
  R_mean = np.nanmean(R_all, axis=0)
/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:107: RuntimeWarning: Mean of empty slice
  R_mean = np.nanmean(R_cat, axis=0)


## 4. Neighbor-based QA

Flag cells whose integrated intensity or peak velocity deviates
from the beam-weighted local plane fit of their neighbors.

Uses `utils.qa.compute_cell_metrics` and `utils.qa.neighbor_qa`.

In [6]:
cell_metrics = compute_cell_metrics(cell_results_combined, v_lsr_overlap, dv_kms)
neighbor_cells = neighbor_qa(cell_metrics, dv_kms=dv_kms)

neighbor_flagged = [c for c in neighbor_cells if c['W_flag'] or c['peak_v_flag']]
neighbor_w_flag_count = sum(1 for c in neighbor_cells if c['W_flag'])
neighbor_peak_flag_count = sum(1 for c in neighbor_cells if c['peak_v_flag'])

print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed')
print(f'  Flags: W={neighbor_w_flag_count}, peak_v={neighbor_peak_flag_count}, '
      f'any={len(neighbor_flagged)}')

if neighbor_flagged:
    print('  Most deviant cells:')

    def severity(cell):
        w_score = abs(cell['W_frac_resid']) if np.isfinite(cell['W_frac_resid']) else 0.0
        v_score = abs(cell['peak_v_z']) if np.isfinite(cell['peak_v_z']) else 0.0
        return max(w_score, v_score)

    for cell in sorted(neighbor_flagged, key=severity, reverse=True)[:12]:
        print(
            f"    l={cell['gl']:3d} b={cell['gb']:3d} "
            f"W={cell['W']:+.3f} (frac={cell['W_frac_resid']:+.2f}, z={cell['W_z']:+.2f}) "
            f"v_peak={cell['peak_v']:+.1f} km/s "
            f"(dv={cell['peak_v_resid']:+.1f}, z={cell['peak_v_z']:+.2f}) "
            f"n={cell['neighbor_count']}"
        )

Neighbor QA: 1636 cells analyzed
  Flags: W=15, peak_v=6, any=21
  Most deviant cells:
    l=125 b=  1 W=+35.796 (frac=+0.14, z=+1.91) v_peak=-56.3 km/s (dv=-39.8, z=-9.32) n=33
    l=114 b= -1 W=+36.746 (frac=+0.05, z=+1.38) v_peak=-47.3 km/s (dv=-19.0, z=-6.33) n=53
    l= 46 b=  0 W=+48.799 (frac=+0.19, z=+1.79) v_peak=+56.1 km/s (dv=+36.9, z=+6.19) n=20
    l=208 b=  0 W=+35.743 (frac=+0.06, z=+0.62) v_peak=+41.4 km/s (dv=+23.1, z=+6.02) n=36
    l=123 b=  1 W=+37.473 (frac=+0.18, z=+4.97) v_peak=-55.7 km/s (dv=-38.6, z=-5.43) n=34
    l=128 b= 16 W=+11.178 (frac=+0.12, z=+6.38) v_peak=-12.5 km/s (dv=-16.0, z=-5.32) n=6
    l=266 b= 24 W=+16.319 (frac=+1.93, z=+19.81) v_peak=-3.5 km/s (dv=+0.6, z=+0.19) n=15
    l= 12 b= -2 W=+16.301 (frac=-0.33, z=-5.23) v_peak=+9.7 km/s (dv=+5.1, z=+1.71) n=5
    l= 12 b= -4 W=+14.137 (frac=+1.46, z=+6.52) v_peak=+5.0 km/s (dv=-3.5, z=-1.18) n=5
    l=250 b= 24 W=+0.257 (frac=-0.95, z=-4.44) v_peak=-6.1 km/s (dv=-1.1, z=-0.35) n=20
    l=242 b= 2

In [7]:
from matplotlib.backends.backend_pdf import PdfPages
from plotters import plot_spectra_grid
import matplotlib.pyplot as plt

with PdfPages('spectra_per_session.pdf') as pdf:
    for dr in sessions:
        dr_spectra = {(l, b): cell_results[(d, l, b)]['R_overlap']
                      for (d, l, b) in cell_results if d == dr}
        if not dr_spectra:
            continue
        result = plot_spectra_grid(v_overlap, dr_spectra,
                                   ncols=5,
                                   color='C0',
                                   title=f'{dr} -- {len(dr_spectra)} pointings')
        figs = result if isinstance(result, list) else [result]
        for f in figs:
            pdf.savefig(f)
            plt.close(f)

print('Saved spectra_per_session.pdf')

Saved spectra_per_session.pdf


## 5. Save reduced data

Serialize the frequency-switched spectra, velocity grid, cell metrics,
QA flags, and per-cell LO dump counts to a single `.npz` file.  Downstream
notebooks (`02a_scan_diagnostics`, `02b_survey_results`) load this file
instead of reprocessing raw dumps.

In [8]:
REDUCED_PATH = Path('../reduced_survey.npz')

# Build arrays for serialization
cell_keys_arr = np.array(list(cell_results_combined.keys()), dtype=int)  # (N, 2)
R_stack = np.array([cell_results_combined[(gl, gb)]['R_overlap']
                     for gl, gb in cell_keys_arr], dtype=float)           # (N, n_ch)
n_pairs_arr = np.array([cell_results_combined[(gl, gb)]['n_pairs']
                          for gl, gb in cell_keys_arr], dtype=int)

# Per-cell LO dump counts (for manifest generation downstream)
lo_counts = {}
for r in records:
    if r.get('gl') is None or r['noise_on']:
        continue
    key = (r['gl'], r['gb'])
    if key not in lo_counts:
        lo_counts[key] = {'n_1420': 0, 'n_1421': 0}
    if r['lo_mhz'] == 1420.0:
        lo_counts[key]['n_1420'] += 1
    elif r['lo_mhz'] == 1421.0:
        lo_counts[key]['n_1421'] += 1

n_1420_arr = np.array([lo_counts.get((gl, gb), {}).get('n_1420', 0)
                        for gl, gb in cell_keys_arr], dtype=int)
n_1421_arr = np.array([lo_counts.get((gl, gb), {}).get('n_1421', 0)
                        for gl, gb in cell_keys_arr], dtype=int)

# QA metrics and flags
metrics_keys = ['W', 'peak_R', 'peak_v', 'peak_prom', 'snr', 'noise_rms',
                'W_frac_resid', 'W_z', 'peak_v_resid', 'peak_v_z',
                'neighbor_count']
metrics_lookup = {(c['gl'], c['gb']): c for c in neighbor_cells}
metrics_arrays = {}
for mk in metrics_keys:
    metrics_arrays[mk] = np.array(
        [metrics_lookup.get((gl, gb), {}).get(mk, np.nan)
         for gl, gb in cell_keys_arr], dtype=float,
    )

W_flag = np.array([metrics_lookup.get((gl, gb), {}).get('W_flag', False)
                    for gl, gb in cell_keys_arr], dtype=bool)
peak_v_flag = np.array([metrics_lookup.get((gl, gb), {}).get('peak_v_flag', False)
                         for gl, gb in cell_keys_arr], dtype=bool)

np.savez_compressed(
    REDUCED_PATH,
    cell_keys=cell_keys_arr,
    R_stack=R_stack,
    v_lsr=v_lsr_overlap,
    dv_kms=dv_kms,
    n_pairs=n_pairs_arr,
    n_1420=n_1420_arr,
    n_1421=n_1421_arr,
    W_flag=W_flag,
    peak_v_flag=peak_v_flag,
    **metrics_arrays,
)
print(f'Saved reduced data: {REDUCED_PATH}')
print(f'  {len(cell_keys_arr)} cells, {R_stack.shape[1]} channels')
print(f'  QA flags: W={W_flag.sum()}, peak_v={peak_v_flag.sum()}')

Saved reduced data: ../reduced_survey.npz
  1636 cells, 418 channels
  QA flags: W=15, peak_v=6


## 6. Summary

In [9]:
all_times = [r['time'] for r in records]
t0 = dt.datetime.fromtimestamp(min(all_times), tz=dt.UTC)
t1 = dt.datetime.fromtimestamp(max(all_times), tz=dt.UTC)

n_sci = sum(1 for r in records if not r['noise_on'])

NBLOCKS = 1025
NSAMPLES = 32768
T_INT = NBLOCKS * NSAMPLES / SAMPLE_RATE_HZ  # 13.107 s per dump
total_int_s = n_sci * T_INT

gl_vals = sorted(set(r['gl'] for r in records if r['gl'] is not None))
gb_vals = sorted(set(r['gb'] for r in records if r['gb'] is not None))

print(f'Scan grid:  {len(gb_vals)} x {len(gl_vals)} = {len(cell_results_combined)} cells')
print(f'LO freqs:   {lo_unique} MHz')
print(f'Start:      {t0:%Y-%m-%d %H:%M:%S UTC}')
print(f'End:        {t1:%Y-%m-%d %H:%M:%S UTC}')
print(f'Integration: {total_int_s/3600:.1f} hr ({n_sci} science dumps x {T_INT:.1f} s)')
print(f'Channels:   {NFFT} (dnu={SAMPLE_RATE_HZ/NFFT/1e3:.2f} kHz)')
print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed')
print(f'  W flags: {neighbor_w_flag_count}, peak_v flags: {neighbor_peak_flag_count}')
print(f'Reduced data: {REDUCED_PATH}')

Scan grid:  34 x 235 = 1636 cells
LO freqs:   [1420.0, 1421.0] MHz
Start:      2026-04-16 09:57:55 UTC
End:        2026-04-24 09:12:52 UTC
Integration: 50.2 hr (13761 science dumps x 13.1 s)
Channels:   1024 (dnu=2.50 kHz)
Neighbor QA: 1636 cells analyzed
  W flags: 15, peak_v flags: 6
Reduced data: ../reduced_survey.npz


## 7. Pointing completeness and duty cycle

In [10]:
# --- Pointing completeness ---
# Complete = >= 3 obs pairs (6 obs dumps across both LOs) AND >= 1 cal pair.
from collections import defaultdict as _defaultdict

_cell_obs = _defaultdict(int)
_cell_cal = _defaultdict(int)
for r in records:
    key = (r['gl'], r['gb'])
    if r['noise_on']:
        _cell_cal[key] += 1
    else:
        _cell_obs[key] += 1

_all_cells = set(_cell_obs) | set(_cell_cal)
_rows = []
for c in sorted(_all_cells):
    obs = _cell_obs.get(c, 0)
    cal = _cell_cal.get(c, 0)
    obs_pairs = obs // 2
    cal_pairs = cal // 2
    _rows.append({
        'l': c[0], 'b': c[1],
        'obs_dumps': obs, 'obs_pairs': obs_pairs,
        'cal_dumps': cal, 'cal_pairs': cal_pairs,
        'complete': obs_pairs >= 3 and cal_pairs >= 1,
    })

completeness_df = pd.DataFrame(_rows)
n_complete = completeness_df['complete'].sum()
n_incomplete = len(completeness_df) - n_complete

summary = pd.DataFrame([
    {'status': 'complete', 'cells': int(n_complete),
     'fraction': f'{n_complete / len(completeness_df):.1%}'},
    {'status': 'incomplete', 'cells': int(n_incomplete),
     'fraction': f'{n_incomplete / len(completeness_df):.1%}'},
])
display(summary)

# Breakdown of incomplete cells by reason
inc = completeness_df[~completeness_df['complete']]
low_obs = (inc['obs_pairs'] < 3).sum()
low_cal = (inc['cal_pairs'] < 1).sum()
both = ((inc['obs_pairs'] < 3) & (inc['cal_pairs'] < 1)).sum()
print(f'Incomplete breakdown: obs_pairs<3: {low_obs}, '
      f'cal_pairs<1: {low_cal}, both: {both}')

,status,cells,fraction
0,complete,19,1.2%
1,incomplete,1619,98.8%


Incomplete breakdown: obs_pairs<3: 18, cal_pairs<1: 1607, both: 6


In [11]:
# --- Duty cycle per session ---
# Integration time per dump from sampling rate: NBLOCKS * NSAMPLES / fs
_int_time = 1025 * 32768 / SAMPLE_RATE_HZ

_duty_rows = []
for s in sessions:
    s_records = [r for r in records if r['session'] == s]
    if len(s_records) < 2:
        continue
    times = sorted(r['time'] for r in s_records)
    n_dumps = len(times)
    wall_s = times[-1] - times[0]
    deltas = np.diff(times)
    median_cadence = float(np.median(deltas))
    duty = min(_int_time / median_cadence, 1.0) * 100
    _duty_rows.append({
        'session': s,
        'dumps': n_dumps,
        'wall_min': round(wall_s / 60, 1),
        'median_cadence_s': round(median_cadence, 1),
        'integration_s': round(_int_time, 1),
        'duty_pct': round(duty, 1),
    })

duty_df = pd.DataFrame(_duty_rows)
display(duty_df)

,session,dumps,wall_min,median_cadence_s,integration_s,duty_pct
0,session_001,1225,601.6,28.5,13.1,46.1
1,session_002,432,217.8,30.3,13.1,43.3
2,session_003,758,372.5,29.5,13.1,44.4
3,session_004,536,150.0,16.9,13.1,77.4
4,session_005,148,41.8,16.9,13.1,77.5
5,session_006,774,211.8,16.4,13.1,79.8
6,session_007,640,178.5,16.7,13.1,78.5
7,session_008,600,160.4,15.9,13.1,82.6
8,session_009,1224,336.7,16.4,13.1,79.8
9,session_010,321,87.4,16.4,13.1,80.0
